<div style="background-color:#000047; padding:30px; border-radius:10px; color:white; text-align:center;">
    <img src='Figures/alinco_white_text.png' style="height:100px; margin-bottom:20px;"/>
    <h1>Módulo 3: Modelos de Lenguaje</h1>
    <h2>Sistema Autocompletado con N-Gramas</h2>
</div>


En este archivo de jupyter construiremos un sistema de autocompletado. Un sistema de autocompletado es algo que probablemente ves todos los días
- Cuando buscas algo en Google, a menudo obtienes sugerencias que te ayudan a completar tu búsqueda.
- Cuando escribes un correo electrónico, recibes sugerencias que te indican posibles finales para tu oración.


<img src = "Figures/stanford.png" style="width:700px;height:300px;"/>



Un componente clave para un sistema de autocompletado es un modelo de lenguaje.
Un modelo de lenguaje asigna la probabilidad a una secuencia de palabras, de manera que las secuencias más "probables" reciben puntuaciones más altas. Por ejemplo, 
>"I have a pen" 
se espera que tenga una probabilidad más alta que 
>"I am a pen"
ya que la primera parece ser una oración más natural en el mundo real.

Puedes aprovechar este cálculo de probabilidad para desarrollar un sistema de autocompletado.  
Supongamos que el usuario escribió 
>"I eat scrambled"
Entonces puedes encontrar una palabra `x` tal que "I eat scrambled x" reciba la probabilidad más alta. Si x = "eggs", la oración sería
>"I eat scrambled eggs"

Si bien se han desarrollado una gran variedad de modelos de lenguaje, esta tarea utiliza **N-gramas**, un método simple pero poderoso para el modelado de lenguaje.
- Los N-gramas también se utilizan en la traducción automática y en el reconocimiento de voz. 


Estos son los pasos de esta tarea:

1. Cargar y preprocesar los datos
    - Cargar y tokenizar los datos.
    - Dividir las oraciones en conjuntos de entrenamiento y prueba.
    - Reemplazar las palabras de baja frecuencia con un marcador de desconocido `<unk>`.
1. Desarrollar modelos de lenguaje basados en N-gramas
    - Calcular el conteo de n-gramas a partir de un conjunto de datos dado.
    - Estimar la probabilidad condicional de la siguiente palabra con suavizado-k (k-smoothing).
1. Evaluar los modelos de N-gramas calculando la puntuación de perplejidad.
1. Usar tu propio modelo para sugerir una palabra siguiente dada tu oración. 

In [1]:
import math
import random
import numpy as np
import pandas as pd
import nltk
nltk.data.path.append('.')

## 1. Cargar y Preprocesar los Datos

Usaremos algunos datos de Twitter. 
Los datos son una cadena larga que contiene muchísimos tweets. Notar que hay un salto de línea "\n" entre los tweets.

In [2]:
with open("en_US.twitter.txt", "r", encoding="utf-8") as f:
    data = f.read()


### Preprocesar los datos

Preprocesaremos los datos de la siguiente manera:

1. Dividir los datos en oraciones usando "\n" como delimitador.
2. Dividir cada oración en tokens (tokenizzación). 
3. Asignar las oraciones a los conjuntos de entrenamiento o prueba.
4. Encontrar los tokens que aparecen al menos N veces en los datos de entrenamiento.
5. Reemplazar los tokens que aparecen menos de N veces por `<unk>`


In [3]:
def split_to_sentences(data):
    sentences = data.split('\n')   
    # pipeline de Limpieza (Esta parte ya está implementada)
    sentences = [s.strip() for s in sentences]
    sentences = [s for s in sentences if len(s) > 0]
    return sentences    

In [6]:
# probando la función
x = """
I have a pen.\nI have an apple. \nAh\nApple pen.\n
"""
split_to_sentences(x)

['I have a pen.', 'I have an apple.', 'Ah', 'Apple pen.']

In [7]:
def tokenize_sentences(sentences):
    tokenized_sentences = []
    # Recorre cada oración
    for sentence in sentences:
        # Convierte a letras minúsculas
        sentence = sentence.lower()
        # Convierte en una lista de palabras
        tokenized = nltk.word_tokenize(sentence)
        # agrega la lista de palabras a la lista de listas
        tokenized_sentences.append(tokenized)
    return tokenized_sentences

In [8]:
# probando la función
sentences = ["Sky is blue.", "Leaves are green.", "Roses are red."]
tokenize_sentences(sentences)

[['sky', 'is', 'blue', '.'],
 ['leaves', 'are', 'green', '.'],
 ['roses', 'are', 'red', '.']]

In [9]:
def get_tokenized_data(data):
    # oraciones dividiendo los datos
    sentences = split_to_sentences(data)
    # lista de listas de tokens tokenizando las oraciones
    tokenized_sentences = tokenize_sentences(sentences)
    return tokenized_sentences

In [10]:
# probando la función
x = "Sky is blue.\nLeaves are green\nRoses are red."
get_tokenized_data(x)

[['sky', 'is', 'blue', '.'],
 ['leaves', 'are', 'green'],
 ['roses', 'are', 'red', '.']]

##### Dividir en conjuntos de entrenamiento y prueba


In [11]:
data

'How are you? Btw thanks for the RT. You gonna be in DC anytime soon? Love to see you. Been way, way too long.\nWhen you meet someone special... you\'ll know. Your heart will beat more rapidly and you\'ll smile for no reason.\nthey\'ve decided its more fun if I don\'t.\nSo Tired D; Played Lazer Tag & Ran A LOT D; Ughh Going To Sleep Like In 5 Minutes ;)\nWords from a complete stranger! Made my birthday even better :)\nFirst Cubs game ever! Wrigley field is gorgeous. This is perfect. Go Cubs Go!\ni no! i get another day off from skool due to the wonderful snow (: and THIS wakes me up...damn thing\nI\'m coo... Jus at work hella tired r u ever in cali\nThe new sundrop commercial ...hehe love at first sight\nwe need to reconnect THIS WEEK\nI always wonder how the guys on the auctions shows learned to talk so fast!? all I hear is djsosnekspqnslanskam.\nDammnnnnn what a catch\nsuch a great picture! The green shirt totally brings out your eyes!\nDesk put together, room all set up. Oh boy, oh 

In [12]:
#Tokenizando los datos cargados
tokenized_data = get_tokenized_data(data)
random.seed(87)
random.shuffle(tokenized_data)

# Entrenamiento y testeo
train_size = int(len(tokenized_data) * 0.8)
train_data = tokenized_data[0:train_size]
test_data = tokenized_data[train_size:]

In [13]:
print(f"El tamaño total de los datos {len(tokenized_data)} dividio en {len(train_data)} para entrenamiento y {len(test_data)} para testeo")


El tamaño total de los datos 47961 dividio en 38368 para entrenamiento y 9593 para testeo


In [14]:
print("First training sample:")
print(train_data[0])
      
print("First test sample")
print(test_data[0])

First training sample:
['i', 'personally', 'would', 'like', 'as', 'our', 'official', 'glove', 'of', 'the', 'team', 'local', 'company', 'and', 'quality', 'production']
First test sample
['that', 'picture', 'i', 'just', 'seen', 'whoa', 'dere', '!', '!', '>', '>', '>', '>', '>', '>', '>']


No usaremos todos los tokens (palabras) que aparecen en los datos para el entrenamiento. En su lugar, usaremos las palabras más frecuentes.  
- Nos enfocaremos en las palabras que aparecen al menos N veces en los datos.
- Primero tendremos que contar cuántas veces aparece cada palabra en los datos.


In [15]:
def count_words(tokenized_sentences):
    word_counts = {}
   
    for sentence in tokenized_sentences: 
        for token in sentence: 
            # Si el token aún no está en el diccionario, establece el conteo en 1
            if token not in word_counts.keys(): # completa esta línea
                word_counts[token] = 1
            # Si el token ya está en el diccionario, incrementa el conteo en 1
            else:
                word_counts[token] += 1
    
    return word_counts

In [16]:
# probando el código
tokenized_sentences = [['sky', 'is', 'blue', '.'],
                       ['leaves', 'are', 'green', '.'],
                       ['roses', 'are', 'red', '.']]
count_words(tokenized_sentences)

{'sky': 1,
 'is': 1,
 'blue': 1,
 '.': 3,
 'leaves': 1,
 'are': 2,
 'green': 1,
 'roses': 1,
 'red': 1}

#### Manejo de palabras 'Fuera del Vocabulario' (Out of Vocabulary)

Si nuestro modelo intenta autoacompletar una oración, pero encuentra una palabra que nunca vio durante el entrenamiento, no tendrá una palabra de entrada que le ayude a determinar la siguiente palabra a sugerir. El modelo no podrá predecir la siguiente palabra porque no hay conteos para la palabra actual. 

- Esta palabra 'nueva' se llama 'palabra desconocida', o palabra <b>fuera del vocabulario (OOV)</b>.
- El porcentaje de palabras desconocidas en el conjunto de prueba se llama tasa <b> OOV </b>. 

Para manejar las palabras desconocidas durante la predicción, usaremos un token especial para representar todas las palabras desconocidas 'unk'. 

- Tendremos que modificar los datos de entrenamiento para que tenga algunas palabras 'desconocidas' con las cuales entrenar.
- Las palabras que se convertirán en palabras "desconocidas" son aquellas que no aparecen con mucha frecuencia en el conjunto de entrenamiento.
- Crearemos una lista de las palabras más frecuentes en el conjunto de entrenamiento, llamada el <b> vocabulario cerrado </b>. 
- Convertiremos todas las demás palabras que no forman parte del vocabulario cerrado al token 'unk'. 


La siguiente función recibe un documento de texto y un umbral 'count_threshold'.

- Cualquier palabra cuyo conteo sea mayor o igual al umbral 'count_threshold' se mantiene en el vocabulario cerrado.
- retorna el documento que contiene solo el vocabulario cerrado de palabras y la palabra unk. 

In [17]:
def get_words_with_nplus_frequency(tokenized_sentences, count_threshold):
    # Inicializa una lista vacía para contener las palabras que
    # aparecen al menos 'minimum_freq' veces.
    closed_vocab = []
    
    # Obtenemos los conteos de palabras de las oraciones tokenizadas
    # Usando la función que definimos antes para contar las palabras
    word_counts = count_words(tokenized_sentences)
    
    # para cada palabra y su conteo
    for word, cnt in word_counts.items(): 
        
        # verificamos que el conteo de la palabra
        # sea al menos tan grande como el conteo mínimo
        if cnt >= count_threshold:
            # agregamos la palabra a la lista
            closed_vocab.append(word)
    
    return closed_vocab

In [18]:
# probamos la función
tokenized_sentences = [['sky', 'is', 'blue', '.'],
                       ['leaves', 'are', 'green', '.'],
                       ['roses', 'are', 'red', '.']]

get_words_with_nplus_frequency(tokenized_sentences, count_threshold= 2)


['.', 'are']

Las palabras que aparecen 'count_threshold' veces o más están en el 'vocabulario cerrado'. 
- Todas las demás palabras las podemos consideran 'desconocidas'.
- La siguiente función reemplaza las palabras que no están en el vocabulario cerrado con el token "<unk\>".

In [19]:
def replace_oov_words_by_unk(tokenized_sentences, vocabulary, unknown_token="<unk>"):
    # Coloca el vocabulario en un conjunto (set) para una búsqueda más rápida
    vocabulary = set(vocabulary)
    
    # Inicializa una lista que contendrá las oraciones
    # después de que las palabras menos frecuentes sean reemplazadas por el token desconocido
    replaced_tokenized_sentences = []
    
    for sentence in tokenized_sentences:
        
        # una sola oración con los reemplazos de "unknown_token"
        replaced_sentence = []

        # para cada token en la oración
        for token in sentence:  
            # Verificamos si el token está en el vocabulario cerrado
            if token in vocabulary:
                # Si es así, agrega la palabra a replaced_sentence
                replaced_sentence.append(token)
            else:
                # de lo contrario, agrega el token desconocido en su lugar
                replaced_sentence.append(unknown_token)
        
        # Agregamos la lista de tokens a la lista de listas
        replaced_tokenized_sentences.append(replaced_sentence)
        
    return replaced_tokenized_sentences

In [20]:
tokenized_sentences = [["dogs", "run"], ["cats", "sleep"]]
vocabulary = ["dogs", "sleep"]

#Probar la función
replace_oov_words_by_unk(tokenized_sentences, vocabulary)

[['dogs', '<unk>'], ['<unk>', 'sleep']]

**Ahora estamos listos para procesar nuestros datos combinando las funciones que acabamos de crear.**

1. La siguiente función encuentra los tokens que aparecen al menos count_threshold veces en los datos de entrenamiento.
1. Reemplaza los tokens que aparecen menos de count_threshold veces por "<unk\>" tanto para los datos de entrenamiento como para los de prueba.

In [21]:
def preprocess_data(train_data, test_data, count_threshold):
    # Obtenemos el vocabulario cerrado usando los datos de entrenamiento
    vocabulary = get_words_with_nplus_frequency(train_data, count_threshold)
        
    # Para los datos de entrenamiento, reemplaza las palabras menos comunes con "<unk>"
    train_data_replaced = replace_oov_words_by_unk(train_data, vocabulary)
    
    # Para los datos de prueba, reemplaza las palabras menos comunes con "<unk>"
    test_data_replaced = replace_oov_words_by_unk(test_data, vocabulary)

    
    return train_data_replaced, test_data_replaced, vocabulary

In [22]:
# probando la función
tmp_train = [['sky', 'is', 'blue', '.'],
     ['leaves', 'are', 'green']]
tmp_test = [['roses', 'are', 'red', '.']]
preprocess_data(tmp_train, tmp_test, count_threshold=1)


([['sky', 'is', 'blue', '.'], ['leaves', 'are', 'green']],
 [['<unk>', 'are', '<unk>', '.']],
 ['sky', 'is', 'blue', '.', 'leaves', 'are', 'green'])

#### Preprocesar los datos de entrenamiento y prueba


In [23]:
#minimo fre = 2
train_data_processed, test_data_processed, vocabulary = preprocess_data(train_data, test_data, count_threshold=2)


In [24]:
train_data_processed

[['i',
  'personally',
  'would',
  'like',
  'as',
  'our',
  'official',
  'glove',
  'of',
  'the',
  'team',
  'local',
  'company',
  'and',
  'quality',
  'production'],
 ['welcome',
  'home',
  '...',
  'and',
  'happy',
  'birthday',
  'president',
  'obama',
  '!'],
 ['i',
  'wish',
  'dvd',
  'manufacturers',
  'would',
  'always',
  'use',
  'the',
  'same',
  'cases',
  ',',
  'so',
  'we',
  'would',
  "n't",
  'have',
  'to',
  'have',
  'a',
  'dozen',
  'kinds',
  'of',
  'security',
  'measures',
  '.'],
 ['where',
  'to',
  'begin',
  '!',
  'i',
  "'ve",
  'been',
  'lucky',
  ',',
  '<unk>',
  'really',
  ',',
  'to',
  'be',
  '<unk>',
  'by',
  'so',
  'many',
  'wonderful',
  'teachers',
  '.'],
 ['thats', 'no', 'fun'],
 ['baby',
  'j',
  '&',
  'at',
  'jordan',
  'ford',
  '<unk>',
  '<unk>',
  'n',
  'w/the',
  'swap',
  'your',
  'ride',
  'offer',
  '!',
  'get',
  'cash',
  '<unk>',
  ',',
  'super',
  'low',
  'financing',
  '&',
  'affordable',
  'monthly

## 2. Desarrollar modelos de lenguaje basados en n-gramas

En esta sección, obtendremos el modelo de lenguaje de n-gramas desde cero.

- Asumimos que la probabilidad de la siguiente palabra depende únicamente del n-grama anterior.
- El n-grama anterior es la serie de las 'n' palabras previas.

**Repaso**

La probabilidad condicional para la palabra en la posición 't' en la oración, dado que las palabras que la preceden son $w_{t-1}, w_{t-2} \cdots w_{t-n}$ es:

$$ P(w_t | w_{t-1}\dots w_{t-n}) \tag{1}$$

Bajo las suposiciones anteriores podemos estimar esta probabilidad contando las apariciones de estas series de palabras en los datos de entrenamiento.
- La probabilidad puede estimarse como una razón, donde
- El numerador es el número de veces que la palabra 't' aparece después de que las palabras t-1 hasta t-n aparecen en los datos de entrenamiento.
- El denominador es el número de veces que las palabras t-1 hasta t-n aparecen en los datos de entrenamiento.

$$ \hat{P}(w_t | w_{t-1}\dots w_{t-n}) = \frac{C(w_{t-1}\dots w_{t-n}, w_n)}{C(w_{t-1}\dots w_{t-n})} \tag{2} $$

- La función $C(\cdots)$ denota el número de apariciones de la secuencia dada. 
- $\hat{P}$ significa la estimación de $P$. 
- Nota que el denominador de la ecuación pasada es el número de apariciones de las $n$ palabras anteriores, y el numerador es la misma secuencia seguida por la palabra $w_t$.

Más adelante, modificarempos esta ecuación agregando suavizado-k (k-smoothing), que evita errores cuando algún conteo es cero.

Basicamente, esta ecuación nos dice que para estimar probabilidades basadas en n-gramas, necesitas los conteos de n-gramas (para el denominador) y de (n+1)-gramas (para el numerador).

`count_n_grams`: Implementaremos una función que calcula los conteos de n-gramas para un número arbitrario $n$.

Al calcular los conteos para los n-gramas, preparaemos la oración de antemano anteponiendo $n-1$ marcadores de inicio "<s\>" para indicar el comienzo de la oración.  
- Por ejemplo, en el modelo de bigramas (N=2), una secuencia con dos tokens de inicio "<s\><s\>" debería predecir la primera palabra de una oración.
- Entonces, si la oración es "I like food", modifícala para que sea "<s\><s\> I like food".
- También prepararemos la oración para el conteo agregando un token de fin "<e\>" para que el modelo pueda predecir cuándo terminar una oración.

En esta implementación, almacenaremos los conteos como un diccionario.
- La clave de cada par clave-valor en el diccionario es una **tupla** de n palabras (y no una lista)
- El valor en el par clave-valor es el número de apariciones.  
- La razón para usar una tupla como clave en lugar de una lista es porque una lista en Python es un objeto mutable (puede cambiarse después de ser creada). Una tupla es "inmutable", por lo que no puede alterarse después de ser creada. Esto hace que una tupla sea adecuada como tipo de dato para la clave en un diccionario.

In [25]:
def count_n_grams(data, n, start_token='<s>', end_token = '<e>'):
    # Inicializa el diccionario de n-gramas y sus conteos
    n_grams = {}

    for sentence in data:
        #agregar el token de inicio <s> n veces, y agregar <e> al final
        sentence = [start_token]*n + sentence + [end_token]

        sentence = tuple(sentence)

        m = len(sentence) if n==1 else len(sentence)-1

        for i in range(m):
            #obtenemos el n-grama en el diccionario
            n_gram = sentence[i:i+n]

            #verificar si el n-grama esta en el diccionario
            if n_gram in n_grams.keys():
                n_grams[n_gram]+=1
            else:
                #inicializamos el conteo de este n-gram en 1
                n_grams[n_gram]=1
    return n_grams

In [26]:
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
#Probamos la función
count_n_grams(sentences, 1, start_token='<s>', end_token = '<e>')

{('<s>',): 2,
 ('i',): 1,
 ('like',): 2,
 ('a',): 2,
 ('cat',): 2,
 ('<e>',): 2,
 ('this',): 1,
 ('dog',): 1,
 ('is',): 1}

In [29]:
count_n_grams(sentences, 2, start_token='<s>', end_token = '<e>')

{('<s>', '<s>'): 2,
 ('<s>', 'i'): 1,
 ('i', 'like'): 1,
 ('like', 'a'): 2,
 ('a', 'cat'): 2,
 ('cat', '<e>'): 2,
 ('<s>', 'this'): 1,
 ('this', 'dog'): 1,
 ('dog', 'is'): 1,
 ('is', 'like'): 1}

A continuación, estima la probabilidad de una palabra dadas las 'n' palabras anteriores usando los conteos de n-gramas.

$$ \hat{P}(w_t | w_{t-1}\dots w_{t-n}) = \frac{C(w_{t-1}\dots w_{t-n}, w_n)}{C(w_{t-1}\dots w_{t-n})} \tag{2} $$

Esta fórmula no funciona cuando el conteo de un n-grama es cero..
- Supongamos que encontramos un n-grama que no apareció en los datos de entrenamiento.  
- Entonces, la ecuación anterior no puede evaluarse (se convierte en cero dividido entre cero).

Una forma de manejar los conteos en cero es agregar suavizado-k (k-smoothing).  
- El suavizado-k agrega una constante positiva $k$ a cada numerador y $k \times |V|$ en el denominador, donde $|V|$ es el número de palabras en el vocabulario.

$$ \hat{P}(w_t | w_{t-1}\dots w_{t-n}) = \frac{C(w_{t-1}\dots w_{t-n}, w_n) + k}{C(w_{t-1}\dots w_{t-n}) + k|V|} \tag{3} $$


Para los n-gramas que tienen un conteo de cero, la ecuación anterior se convierte en $\frac{1}{|V|}$.
- Esto significa que cualquier n-grama con conteo cero tiene la misma probabilidad de $\frac{1}{|V|}$.

Define una función que calcule la estimación de probabilidad anterior a partir de los conteos de n-gramas y una constante $k$.

- La función recibe un diccionario 'n_gram_counts', donde la clave es el n-grama y el valor es el conteo de ese n-grama.
- La función también recibe otro diccionario n_plus1_gram_counts, que usarás para encontrar el conteo del n-grama anterior más la palabra actual.

In [30]:
def estimate_probability(word, previous_n_gram, 
                         n_gram_counts, n_plus1_gram_counts, vocabulary_size, k=1.0):
    
    # convierte la lista en tupla para usarla como clave de diccionario
    previous_n_gram = tuple(previous_n_gram)
    
    # Establecemos el denominador
    # Si el n-grama anterior existe en el diccionario de conteos de n-gramas,
    # Obtenemos su conteo. De lo contrario establece el conteo en cero
    # Usamos el diccionario que tiene los conteos de n-gramas
    previous_n_gram_count = n_gram_counts[previous_n_gram] if previous_n_gram in n_gram_counts else 0
        
    # Calcularemos el denominador usando el conteo del n-grama anterior
    # y aplicamos suavizado-k
    
    denominator = previous_n_gram_count + k*vocabulary_size
    # Definamos el (n+1)-grama como el n-grama anterior más la palabra actual como una tupla
    n_plus1_gram = previous_n_gram + (word,)
    
    # Establecemos el conteo al conteo en el diccionario,
    # de lo contrario 0 si no está en el diccionario
    # usamos el diccionario que tiene los conteos del n-grama más la palabra actual
    n_plus1_gram_count = n_plus1_gram_counts[n_plus1_gram] if n_plus1_gram in n_plus1_gram_counts else 0

    # Definimos el numerador usando el conteo del n-grama más la palabra actual,
    # y aplicamos el suavizado
    numerator = n_plus1_gram_count + k

    # Calculamos la probabilidad como el numerador dividido entre el denominador
    probability = numerator/denominator
    
    return probability

In [32]:
# probamos la función
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)



In [34]:
estimate_probability('cat', 'a',unigram_counts, bigram_counts, len(unique_words), k=1.0)

0.3333333333333333

#### Estimar probabilidades para todas las palabras

La función definida a continuación recorre todas las palabras del vocabulario para calcular las probabilidades de todas las palabras posibles.


In [35]:
def estimate_probabilities(previous_n_gram, n_gram_counts, n_plus1_gram_counts, vocabulary, k=1.0):
    
    # convierte la lista en tupla para usarla como clave de diccionario
    previous_n_gram = tuple(previous_n_gram)
    
    # agrega <e> <unk> al vocabulario
    # <s> no es necesario ya que no debería aparecer como la siguiente palabra
    vocabulary = vocabulary + ["<e>", "<unk>"]
    vocabulary_size = len(vocabulary)
    
    probabilities = {}
    for word in vocabulary:
        probability = estimate_probability(word, previous_n_gram, 
                                           n_gram_counts, n_plus1_gram_counts, 
                                           vocabulary_size, k=k)
        probabilities[word] = probability

    return probabilities

In [36]:
# probamos la función
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)


In [37]:
estimate_probabilities('a', unigram_counts, bigram_counts, unique_words, k=1)

{'i': 0.09090909090909091,
 'cat': 0.2727272727272727,
 'like': 0.09090909090909091,
 'a': 0.09090909090909091,
 'is': 0.09090909090909091,
 'this': 0.09090909090909091,
 'dog': 0.09090909090909091,
 '<e>': 0.09090909090909091,
 '<unk>': 0.09090909090909091}

In [38]:
# Prueba adicional
trigram_counts = count_n_grams(sentences, 3)
estimate_probabilities(['<s>', '<s>'], bigram_counts, trigram_counts, unique_words, k=1)


{'i': 0.18181818181818182,
 'cat': 0.09090909090909091,
 'like': 0.09090909090909091,
 'a': 0.09090909090909091,
 'is': 0.09090909090909091,
 'this': 0.18181818181818182,
 'dog': 0.09090909090909091,
 '<e>': 0.09090909090909091,
 '<unk>': 0.09090909090909091}

#### Matrices de conteo y de probabilidad

Como hemos visto hasta ahora, los conteos de n-gramas calculados arriba son suficientes para calcular las probabilidades de la siguiente palabra.  
- Puede ser más intuitivo presentarlos como matrices de conteo o de probabilidad.
- Las funciones definidas en las siguientes celdas retornan matrices de conteo o de probabilidad.
- Esta función se te proporciona.

In [ ]:
def make_count_matrix(n_plus1_gram_counts, vocabulary):
    # agregamos <e> <unk> al vocabulario
    # <s> se omite ya que no debería aparecer como la siguiente palabra
    vocabulary = vocabulary + ["<e>", "<unk>"]
    
    # obtenemos los n-gramas únicos
    n_grams = []
    for n_plus1_gram in n_plus1_gram_counts.keys():
        n_gram = n_plus1_gram[0:-1]
        n_grams.append(n_gram)
    n_grams = list(set(n_grams))
    
    # mapeamos de n-grama a fila
    row_index = {n_gram:i for i, n_gram in enumerate(n_grams)}
    # mapeamos de la siguiente palabra a columna
    col_index = {word:j for j, word in enumerate(vocabulary)}
    
    nrow = len(n_grams)
    ncol = len(vocabulary)
    count_matrix = np.zeros((nrow, ncol))
    for n_plus1_gram, count in n_plus1_gram_counts.items():
        n_gram = n_plus1_gram[0:-1]
        word = n_plus1_gram[-1]
        if word not in vocabulary:
            continue
        i = row_index[n_gram]
        j = col_index[word]
        count_matrix[i, j] = count
    
    count_matrix = pd.DataFrame(count_matrix, index=n_grams, columns=vocabulary)
    return count_matrix

In [ ]:
sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
bigram_counts = count_n_grams(sentences, 2)

print('bigram counts')
display(make_count_matrix(bigram_counts, unique_words))

In [ ]:
# Mostramos los conteos de trigramas
print('\ntrigram counts')
trigram_counts = count_n_grams(sentences, 3)
display(make_count_matrix(trigram_counts, unique_words))

La siguiente función calcula las probabilidades de cada palabra dado el n-grama anterior, y las almacena en forma de matriz.
- Esta función se te proporciona.

In [ ]:
def make_probability_matrix(n_plus1_gram_counts, vocabulary, k):
    count_matrix = make_count_matrix(n_plus1_gram_counts, unique_words)
    count_matrix += k
    prob_matrix = count_matrix.div(count_matrix.sum(axis=1), axis=0)
    return prob_matrix

In [ ]:
sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
bigram_counts = count_n_grams(sentences, 2)
print("bigram probabilities")
display(make_probability_matrix(bigram_counts, unique_words, k=1))

In [ ]:
print("trigram probabilities")
trigram_counts = count_n_grams(sentences, 3)
display(make_probability_matrix(trigram_counts, unique_words, k=1))

## 3. Perplejidad

En esta sección, generaremos la puntuación de perplejidad para evaluar tu modelo en el conjunto de prueba. 
- También usaremos retroceso (back-off) cuando sea necesario. 
- La perplejidad se usa como métrica de evaluación de tu modelo de lenguaje. 
- Para calcular la puntuación de perplejidad del conjunto de prueba en un modelo de n-gramas, usa: 

$$ PP(W) =\sqrt[N]{ \prod_{t=n+1}^N \frac{1}{P(w_t | w_{t-n} \cdots w_{t-1})} } \tag{4}$$

- donde $N$ es la longitud de la oración.
- $n$ es el número de palabras en el n-grama (por ejemplo, 2 para un bigrama).
- En matemáticas, la numeración empieza en uno y no en cero.

En código, la indexación de arreglos empieza en cero, por lo que el código usará rangos para $t$ según esta fórmula:

$$ PP(W) =\sqrt[N]{ \prod_{t=n}^{N-1} \frac{1}{P(w_t | w_{t-n} \cdots w_{t-1})} } \tag{4.1}$$

Mientras más altas sean las probabilidades, menor será la perplejidad. 
- Mientras más nos digan los n-gramas sobre la oración, menor será la puntuación de perplejidad. 

**Calculamos la puntuación de perplejidad dada una matriz de conteo de N-gramas y una oración.**

In [ ]:
def calculate_perplexity(sentence, n_gram_counts, n_plus1_gram_counts, vocabulary_size, k=1.0):
    
    # longitud de las palabras anteriores
    
    
    # anteponemos <s> y agrega <e>
    
    
    # Convertimos la oración de una lista a una tupla
    
    
    # longitud de la oración (después de agregar los tokens <s> y <e>)
    
    
    # La variable p contendrá el producto
    # que se calcula dentro de la raíz n-ésima
    # Actualiza esto en el código de abajo
    
    
    # El índice t va de n a N - 1, inclusivo en ambos extremos
    

        # obtenemos el n-grama que precede a la palabra en la posición t
    
        
        # obtenemos la palabra en la posición t
    
        
        # Estimamos la probabilidad de la palabra dado el n-grama
        # usando los conteos de n-gramas, los conteos de (n+1)-gramas,
        # el tamaño del vocabulario y la constante de suavizado
    
        
        # Actualizamos el producto de las probabilidades
        # Este 'product_pi' es un producto acumulativo 
        # de los factores (1/P) que se calculan en el bucle
        

    # Tomamos la raíz N-ésima del producto
    
    
    return None

In [ ]:
# probando la función

sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)



In [ ]:
#calculate_perplexity



In [ ]:
#calculate_perplexity test sentence
test_sentence = ['i', 'like', 'a', 'dog']


<b> Nota: </b> Si la oración es muy larga, habrá desbordamiento por defecto (underflow) al multiplicar muchas fracciones.
- Para manejar oraciones más largas, modificaremos esta función para tomar la suma del logaritmo de las probabilidades.

<a name='4'></a>
## 4. Construyendo un sistema de autocompletado

En esta sección, combinaremos los modelos de lenguaje desarrollados hasta ahora para implementar un **sistema de autocompletado**. 


Calcularemos las probabilidades para todas las posibles siguientes palabras y sugiriremos la más probable.
- Esta función también recibe un argumento opcional `start_with`, que especifica las primeras letras de las siguientes palabras.

In [ ]:
def suggest_a_word(previous_tokens, n_gram_counts, n_plus1_gram_counts, vocabulary, k=1.0, start_with=None):
    
    # longitud de las palabras anteriores
    n = len(list(n_gram_counts.keys())[0]) 
    
    # De las palabras que el usuario ya escribió
    # obtenemos las 'n' palabras más recientes como el n-grama anterior
    previous_n_gram = previous_tokens[-n:]

    # Estimamos las probabilidades de que cada palabra en el vocabulario
    # sea la siguiente palabra,
    # dado el n-grama anterior, el diccionario de conteos de n-gramas,
    # el diccionario de conteos de (n+1)-gramas, y la constante de suavizado
    probabilities = estimate_probabilities(previous_n_gram,
                                           n_gram_counts, n_plus1_gram_counts,
                                           vocabulary, k=k)
    
    # Inicializamos la palabra sugerida en None
    # Esto establecerá a la palabra con la probabilidad más alta
    suggestion = None
    
    # Inicializamos la probabilidad de palabra más alta en 0
    # esto establecerá a la probabilidad más alta 
    # de todas las palabras a sugerir
    max_prob = 0
    
    # Para cada palabra y su probabilidad en el diccionario de probabilidades:
    for word, prob in probabilities.items(): 
        
        # Si la cadena opcional start_with está establecida
        if start_with != None:
            
            # Verificamos si el inicio de la palabra no coincide con las letras en 'start_with'
            if not word.startswith(start_with):

                # si no coinciden, omite esta palabra (pasa a la siguiente palabra)
                continue  
        
        # Verificamos si la probabilidad de esta palabra
        # es mayor que la probabilidad máxima actual
        if prob > max_prob: 
            
            # Si es así, guarda esta palabra como la mejor sugerencia (hasta ahora)
            suggestion = word
            
            # Guarda la nueva probabilidad máxima
            max_prob = prob

    return suggestion, max_prob

In [ ]:
# prueba tu código
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)


In [ ]:
previous_tokens = ["i", "like"]




In [ ]:
# probemos el código estableciendo el starts_with
tmp_starts_with = 'c'


#### Obtener múltiples sugerencias

La función definida a continuación recorre varios modelos de n-gramas para obtener múltiples sugerencias.

In [ ]:
def get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0, start_with=None):
    model_counts = len(n_gram_counts_list)
    suggestions = []
    for i in range(model_counts-1):
        n_gram_counts = n_gram_counts_list[i]
        n_plus1_gram_counts = n_gram_counts_list[i+1]
        
        suggestion = suggest_a_word(previous_tokens, n_gram_counts,
                                    n_plus1_gram_counts, vocabulary,
                                    k=k, start_with=start_with)
        suggestions.append(suggestion)
    return suggestions

In [ ]:
# probamos la función
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)
trigram_counts = count_n_grams(sentences, 3)
quadgram_counts = count_n_grams(sentences, 4)
qintgram_counts = count_n_grams(sentences, 5)

n_gram_counts_list = [unigram_counts, bigram_counts, trigram_counts, quadgram_counts, qintgram_counts]
previous_tokens = ["i", "like"]

In [ ]:
#sugiriendo la próxima palabra


#### Sugerir múltiples palabras usando n-gramas de longitud variable

Veámos como sugerir múltiples palabras usando n-gramas de longitudes variables (unigramas, bigramas, trigramas, 4-gramas...6-gramas).

In [ ]:
n_gram_counts_list = []
for n in range(1, 6):
    print("Calculando conteo de n-gram con n =", n, "...")
    n_model_counts = count_n_grams(train_data_processed, n)
    n_gram_counts_list.append(n_model_counts)

In [ ]:
previous_tokens = ["i", "am", "to"]


In [ ]:
previous_tokens = ["i", "want", "to", "go"]


In [ ]:
previous_tokens = ["hey", "how", "are"]


In [ ]:
previous_tokens = ["hey", "how", "are", "you"]


In [ ]:
previous_tokens = ["hey", "how", "are", "you"]
